# Iktos 3D Align API - AlignDockBench Notebook

This notebook runs the Iktos 3D Align API on AlignDockBench benchmark dataset of protein/reference-ligand/query-ligand triplets, and computes RMSD between predicted and ground-truth poses.

**Workflow overview:**
1. Initialize the API client
2. Check API quota
3. Load the benchmark and prepare inputs
5. Submit prediction jobs (one per benchmark entry)
6. Monitor progress and retrieve results
7. Compute RMSD and save a results dataframe

## 1. Setup

In [ ]:
import os

import pandas as pd
from rdkit import Chem
from posebusters.modules.rmsd import check_rmsd

from align_3d_client import (
    Align3DClient,
    ReferenceLigandInput,
    LigandInput,
)

In [ ]:
API_URL = os.getenv("API_URL", "https://dashboard.engine.iktos.ai/v1/3d-align/")
API_KEY = os.getenv("API_KEY", "YOUR_API_KEY")

client = Align3DClient(API_URL, api_key=API_KEY)

## 2. Quota Check

In [ ]:
quota = client.get_quota()
print(f"Quota: {quota.quota_used}/{quota.quota_max} used, {quota.quota_remaining} remaining")

## 3. Load Benchmark Dataset & Prepare inputs

Download the AlignDockBench dataset from [Zenodo](https://zenodo.org/records/15395813), then set `DATA_DIR` to the path of the extracted folder.

### Dataset structure

```
AlignDockBench/
└── {id_a}_{ligand_name}_{chain}_{resnum}/   ← entry (batch_job_id)
    ├── {id_a}/
    │   └── ligand.sdf                        ← reference ligand (crystallographic pose of id_a)
    ├── {id_b_1}/
    │   └── ligand.sdf                        ← query 1 (ground-truth pose)
    ├── {id_b_2}/
    │   └── ligand.sdf                        ← query 2 (ground-truth pose)
    └── ...
```

- **`id_a`** — PDB ID of the reference ligand, extracted from the entry folder name (`entry.split("_")[0]`)
- **`id_b_*`** — PDB IDs of query ligands (all other subfolders in the entry)
- 61 entries, ~6 query ligands per entry on average (369 total)

**What is sent to the API**

| Field | Source | Format |
|---|---|---|
| `ReferenceLigandInput` | `id_a/ligand.sdf` → mol_block | Full 3D SDF of the reference (crystallographic pose) |
| `LigandInput` (query) | `id_b/ligand.sdf` → SMILES only | 2D SMILES of the query (no 3D coordinates) |

The query is intentionally degraded to SMILES to simulate the real use case: 2D structure known, 3D pose unknown.

**Evaluation**

RMSD is computed between the API-predicted pose and the ground-truth `id_b/ligand.sdf` using `posebusters.check_rmsd`.

In [ ]:
DATA_DIR = "path/to/AlignDockBench"

In [ ]:

def build_entry_inputs(entry: str):
    id_a = entry.split("_")[0]

    items = os.listdir(os.path.join(DATA_DIR, entry))
    ids_b = [x for x in items if x != id_a and x != ".DS_Store"]

    # Reference ligand (from id_a subfolder)
    ref_mol = Chem.MolFromMolFile((os.path.join(DATA_DIR, entry, id_a, "ligand.sdf")))
    reference = ReferenceLigandInput(
        id=id_a,
        mol_block=Chem.MolToMolBlock(ref_mol),
    )

    # Query ligands: one LigandInput per id_b
    ligands = []
    for id_b in ids_b:
        query_mol = Chem.MolFromMolFile(os.path.join(DATA_DIR, entry, id_b, "ligand.sdf"))
        if query_mol is None:
            print(f"  [warn] could not parse SDF for {entry}/{id_b}, skipping")
            continue
        ligands.append(LigandInput(id=id_b, smiles=Chem.MolToSmiles(query_mol)))

    return reference, ligands, entry

entries_a = [id for id in os.listdir(DATA_DIR) if id != ".DS_Store"]
inputs = [build_entry_inputs(e) for e in entries_a]
total_ligands = sum(len(ligs) for _, ligs, _ in inputs)
print(f"Prepared {len(inputs)} entries ({total_ligands} total ligands).")

## 4. Submit Prediction Jobs


In [ ]:
request_ids = {}

for reference, ligands, batch_job_id in inputs:
    response = client.create_request(
        reference_ligand=reference,
        ligands=ligands,
        batch_job_id=batch_job_id,
    )
    request_ids[batch_job_id] = response["request_id"]

print(f"Submitted {len(request_ids)} requests.")

## 5. Monitor Progress & Retrieve results


In [ ]:
def show_progress(request):
    s = request.status_summary
    if s is not None:
        print(f"  {request.batch_job_id}: {s.completed}/{s.total} completed, {s.failed} failed")


final_status = {}
for batch_job_id, request_id in request_ids.items():
    final_status[batch_job_id] = client.wait_for_completion(
        request_id,
        check_interval=5,
        progress_callback=None,  # set to show_progress for verbose polling
    )

print(f"All {len(final_status)} requests reached a terminal state.")

Each completed job returns:
- `pharmaco_score`
- `shape_score`
- `ligand_smiles`
- `ligand_pose_sdf` (predicted 3D pose)

In [ ]:
# Nested dict: batch_job_id -> ligand_id -> { pose_sdf, time_elapsed, scores, status, error }
results_by_entry = {}

for batch_job_id, request_id in request_ids.items():
    ligand_results = client.get_results(request_id)
    per_ligand = {}
    for ligand_id, res in ligand_results.items():
        if res["status"] == "COMPLETED":
            data = res["data"]
            per_ligand[ligand_id] = {
                "status": "COMPLETED",
                "pose_sdf": data.get("ligand_pose_sdf"),
                "pharmaco_score": data.get("pharmaco_score"),
                "shape_score": data.get("shape_score"),
                "time_elapsed": res.get("time_elapsed"),
            }
        else:
            per_ligand[ligand_id] = {
                "status": res["status"],
                "pose_sdf": None,
                "pharmaco_score": None,
                "shape_score": None,
                "time_elapsed": res.get("time_elapsed"),
                "error": res.get("error"),
            }
    results_by_entry[batch_job_id] = per_ligand

# Summary
total_ligands = sum(len(v) for v in results_by_entry.values())
completed_ligands = sum(
    1
    for per_ligand in results_by_entry.values()
    for r in per_ligand.values()
    if r["status"] == "COMPLETED"
)
print(f"Completed: {completed_ligands}/{total_ligands} ligands across {len(results_by_entry)} entries.")

## 7. Compute RMSD and Save

RMSD is computed with `posebusters.modules.rmsd.check_rmsd` against the crystallo molecule.

In [ ]:
rows = []
for reference, ligands, batch_job_id in inputs:
    per_ligand = results_by_entry.get(batch_job_id, {})

    for lig in ligands:
        ligand_id = lig.id  # = id_b
        res = per_ligand.get(ligand_id, {})
        pose_sdf = res.get("pose_sdf")

        rmsd = None
        if pose_sdf is not None:
            try:
                gt_mol = Chem.MolFromMolFile(
                    (os.path.join(DATA_DIR, batch_job_id, ligand_id, "ligand.sdf"))
                )
                predicted_mol = Chem.MolFromMolBlock(pose_sdf)
                rmsd = check_rmsd(predicted_mol, gt_mol)["results"]["rmsd"]
            except Exception:
                rmsd = None

        rows.append({
            "batch_job_id": batch_job_id,
            "ligand_id": ligand_id,
            "query_smiles": lig.smiles,
            "status": res.get("status"),
            "rmsd": rmsd,
            "time_elapsed": res.get("time_elapsed"),
            "pharmaco_score": res.get("pharmaco_score"),
            "shape_score": res.get("shape_score"),
        })

df_results = pd.DataFrame(rows)
df_results.head()

In [ ]:
PATH_SAVE_RESULTS = "results/3DAlign_benchmark_results.csv"
df_results.to_csv(PATH_SAVE_RESULTS, index=False)
print(f"Saved {len(df_results)} rows to {PATH_SAVE_RESULTS}")